# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ranahassan9/Machine-Learning-FlyRank-AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



### 1. My lane as an ML task (type)

**ML Task Type:** Multi-Class Supervised Classification

We frame our Dual-Engine Defense pipeline as a 6-class supervised classification problem. Rather than predicting a simple binary "yes/no" traffic drop, our model predicts a specific operational action category for every web page. The model takes tabular features derived from Google Search Console (GSC), Google Analytics (GA4), and page-level metadata (`word_count`, `main_intent`) to map each page directly to its optimal recovery or protection pathway.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




### 2. Target or proxy

**Target Variable:** `recommended_action` (Categorical)

Because operational action tags are not pre-labeled in raw analytics logs, we create a structured proxy target combining observed traditional search decay with conversational AI traffic guardrails:

* **Protect & Lock (AI Active):** `ai_sessions_90d > 0` $\rightarrow$ Page earns direct conversational AI referrals. Do not alter text or layout.
* **Update Meta Tags & Snippet:** Declining (`is_declining_label == 1`) + High Visibility (`impressions_90d > 2000`) + Low CTR (`ctr < 0.05`) $\rightarrow$ Rewrite title and description snippets to improve click-through rate.
* **Expand Content Depth:** Declining + Striking Distance Rank (`10 <= avg_position <= 25`) $\rightarrow$ Add subtopics, FAQs, and missing target keywords to push into top 10 rankings.
* **Full Editorial Overhaul:** Declining + Severe Organic Decay (`trend_pct < -50`) $\rightarrow$ Reinvest editorial budget for a complete content rewrite.
* **Prune or Consolidate:** Declining + Does not meet CTR, rank, or severe drop thresholds $\rightarrow$ De-index, delete, or 301-redirect low-performing pages.
* **Maintain & Monitor:** Non-declining pages (`is_declining_label == 0`) $\rightarrow$ Retain existing structure and monitor standard performance.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




### 3. Success metric

**Primary Evaluation Metric:** Macro F1-Score (supported by Class-specific Recall for high-risk categories).

**Why Macro F1-Score?** Accuracy is misleading because class distributions are imbalanced (e.g., only 6.4% of pages have AI traffic, while 43% are in Maintain & Monitor). Macro F1 treats all six action classes equally, ensuring the model performs reliably across both rare and common categories.

**Business-Critical Sub-Metrics:**

* **High Recall on Protect & Lock (> 90%):** Ensures we rarely misclassify an active AI traffic driver as a page that needs an invasive rewrite (minimizing False Positives that break AI referrals).
* **High Precision on Full Editorial Overhaul (> 85%):** Ensures we do not waste editorial rewrite budgets on pages that do not actually need major structural intervention.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




### 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** **One row = One unique web page (`content_hash_id`) evaluated over a 90-day observation window.**

Run this code cell in Section 4 to load the starter dataset, construct the unit of analysis dataframe, sketch the target column, and display the class distribution:



In [4]:
import pandas as pd
import numpy as np

# Load dataset
csv_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_url)

# Define binary decline indicator using exact column 'trend_direction'
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)

# 1. Define proxy target assignment using exact dataset columns
def assign_action_proxy(row):
    # Rule 1: AI Guardrail First
    if row.get('ai_sessions_90d', 0) > 0:
        return 'Protect & Lock (AI Active)'

    # Rule 2: Actions for Declining Pages
    if row.get('is_declining_label', 0) == 1:
        # Check CTR in decimal format (e.g., 0.10 = 10%)
        if row.get('impressions_90d', 0) > 2000 and row.get('ctr', 0) < 0.05:
            return 'Update Meta Tags & Snippet'
        elif 10 <= row.get('avg_position', 0) <= 25:
            return 'Expand Content Depth'
        elif row.get('trend_pct', 0) < -50:
            return 'Full Editorial Overhaul'
        else:
            return 'Prune or Consolidate'

    return 'Maintain & Monitor'

# 2. Generate target column
df['recommended_action'] = df.apply(assign_action_proxy, axis=1)

# 3. Display Unit of Analysis Dataframe
unit_of_analysis_cols = [
    'content_id',
    'main_intent',
    'word_count',
    'impressions_90d',
    'ctr',
    'avg_position',
    'ai_sessions_90d',
    'recommended_action'
]

unit_of_analysis_df = df[unit_of_analysis_cols]

print(f"Total Unit of Analysis Rows (Unique Pages): {len(unit_of_analysis_df):,}\n")
print("--- Dataframe Preview (Unit of Analysis) ---")
print(unit_of_analysis_df.head())

print("\n--- Target Column Class Distribution ---")
print(unit_of_analysis_df['recommended_action'].value_counts())

Total Unit of Analysis Rows (Unique Pages): 30,000

--- Dataframe Preview (Unit of Analysis) ---
             content_id    main_intent  word_count  impressions_90d   ctr  \
0  content_304f48230142  transactional      3221.0             3803  0.76   
1  content_a1fb4e703a9e  informational      2481.0            15320  0.05   
2  content_9aa793d4d895  informational      3515.0            12581  0.09   
3  content_331d6c4de07b     commercial         NaN            11751  0.49   
4  content_d99b7a2d90ca  informational      2803.0            19140  0.13   

   avg_position  ai_sessions_90d       recommended_action  
0          10.6                0     Expand Content Depth  
1          20.3                0     Expand Content Depth  
2          36.5                0  Full Editorial Overhaul  
3           6.2                0       Maintain & Monitor  
4          44.0                0     Prune or Consolidate  

--- Target Column Class Distribution ---
recommended_action
Maintain & Monitor 



### 5. Why ML beats a fixed rule here

* **Multi-Dimensional Signal Interactivity:** Simple IF-THEN rules evaluate metrics in isolation (e.g., IF `word_count < 500`). Machine learning models evaluate complex non-linear interactions across 10+ signals simultaneously (e.g., how ranking position, engagement rate, CTR, and intent interact together).
* **Automated Scalability Across Enterprise Portfolios:** Manual inspection by an SEO specialist takes ~15 minutes per page, capping output at ~100 pages a week. An ML pipeline classifies 30,000+ pages in seconds.
* **Predictive Early Warning System:** Static human rules react after a page crashes. Tree-based ML models learn subtle leading indicators—such as creeping position shifts paired with CTR decay—to predict and flag drops before severe impression loss occurs.
* **Probabilistic Budget Optimization:** Instead of binary cutoffs, ML outputs continuous prediction probabilities, enabling content teams to rank pages by urgency and spend their rewrite budget strictly on pages with the highest probability of recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.